In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-18 19:47:47,527 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b
2026-04-18 19:47:47,671 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: openai


In [3]:
test_invoke_without_tool(agent)

2026-04-18 19:47:52,699 | INFO | 对话历史已清空
2026-04-18 19:47:52,700 | INFO | 使用普通模式调用智能体
2026-04-18 19:48:01,003 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"




你好，我是你的 AI 助手。

我主要能帮你做这些事：
- 回答技术与逻辑问题
- 编写、调试或解释代码
- 分析文档或处理数据
- 辅助完成工程任务

有什么具体需求，直接告诉我。


In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [4]:
await test_astream_without_tool(agent)

2026-04-18 19:48:05,910 | INFO | 对话历史已清空
2026-04-18 19:48:05,940 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


thinking content:
Thinking Process:

1.  **Analyze the Request:**
    *   User: "你好，请介绍一下你自己" (Hello, please introduce yourself).
    *   Intent: The user is asking for a self-introduction.
    *   Context: I am an AI assistant following specific system interaction rules (GitHub-style Markdown, direct communication, safety, efficiency, etc.).
    *   Constraint: I need to adhere to the system instructions provided in the prompt (rules about text output, tool usage, Markdown style, task execution, risk/security, tone, and efficiency).

2.  **Review System Instructions:**
    *   **Role:** Useful AI assistant.
    *   **Output:** Direct communication to the user (not internal drafts).
    *   **Format:** GitHub-style Markdown.
    *   **Tone:** Direct, clear, restrained, no excessive emojis or padding.
    *   **Efficiency:** Action/conclusion first, then explanation if needed.
    *   **Security:** Avoid obvious security issues.
    *   **Task Execution:** Prioritize actual work if requ

In [5]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-18 19:48:17,108 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-18 19:48:17,109 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-18 19:48:17,110 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-18 19:48:17,110 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [6]:
test_invoke_with_tool(agent)

2026-04-18 19:48:19,155 | INFO | 对话历史已清空
2026-04-18 19:48:19,156 | INFO | 使用工具模式调用智能体
2026-04-18 19:48:21,090 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 19:48:21,092 | INFO | 思考内容: 用户有两个请求：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

我需要先使用工具来完成这两个任务，然后判断翻译工具是否正确。

让我先调用翻译工具和计算器工具。

2026-04-18 19:48:21,093 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-18 19:48:21,094 | INFO | test_skill执行工具: calculator，参数: {'expression': '3^22'}
2026-04-18 19:48:22,662 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 19:48:22,664 | INFO | 思考内容: 翻译工具返回的结果有问题，它没有正确翻译中文到英文。让我再试一次翻译。

计算器的结果 3^22 = 31381059609 是正确的。

让我重新翻译。

2026-04-18 19:48:22,665 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-18 19:48:26,164 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-18 19:48:26,165 | I



### 翻译工具状态：❌ 异常

翻译工具未能正确执行，返回的结果仍是中文原文：
- 输入：`你是谁，在哪里`
- 工具输出：`你是谁，在哪里`（未翻译）

这表明翻译工具当前可能未正确配置或存在故障。

### 数学计算结果：✅ 正确

- 表达式：`3^22`
- 结果：**31,381,059,609**
- 验证：3的22次方计算无误。

需要我尝试其他方式翻译该句子吗？


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [7]:
await test_astream_with_tool(agent)

2026-04-18 19:48:29,956 | INFO | 对话历史已清空
2026-04-18 19:48:29,983 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我：
1. 翻译"你是谁，在哪里"到英语
2. 计算3^22

我需要：
1. 使用 translate_tool 翻译文本到英语
2. 使用 calculator 计算 3^22

这两个任务可以并行执行，因为它们互不依赖。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3^22'}


2026-04-18 19:48:32,100 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 2

thinking content:
翻译工具似乎没有正确翻译中文到英文，只是原样返回了中文文本。计算器工具正确计算了 3^22 = 31,381,059,609。

让我重新翻译，确认工具是否正常工作。

content:


翻译工具似乎没有正常工作，它原样返回了中文文本而没有翻译成英文。让我重新尝试翻译：


tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}


2026-04-18 19:48:34,027 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 3

thinking content:
翻译工具似乎无法正常工作，它没有将中文翻译成英文，而是原样返回了中文文本。

计算器工作正常，3^22 = 31381059609。

我直接告诉用户翻译工具的问题。

content:


### 翻译工具状态：**工作异常**

翻译工具未能正确执行翻译任务，它原样返回了中文文本 "你是谁，在哪里"，而没有翻译成英文。这是一个工具问题，而非输入问题。

### 计算器结果：**正常**

- **3^22 = 31,381,059,609**

---

### 正确的翻译（由我提供）：
**"Who are you, where are you"** 或更自然的表达：
- **"Who are you and where are you?"**
- **"Who are you and where are you located?"**

需要我继续用其他方式协助你吗？
final res:


### 翻译工具状态：**工作异常**

翻译工具未能正确执行翻译任务，它原样返回了中文文本 "你是谁，在哪里"，而没有翻译成英文。这是一个工具问题，而非输入问题。

### 计算器结果：**正常**

- **3^22 = 31,381,059,609**

---

### 正确的翻译（由我提供）：
**"Who are you, where are you"** 或更自然的表达：
- **"Who are you and where are you?"**
- **"Who are you and where are you located?"**

需要我继续用其他方式协助你吗？


In [ ]:
raw_history=agent.get_raw_history()

In [ ]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history==raw_history2

In [ ]:
await agent.astream_invoke(f"在帮我翻译一下:我是一直小白马")


In [ ]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

In [ ]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

In [ ]:
await agent.astream_invoke(f"我们刚才聊了什么")
